# Milestone 1 — Tabular MLP

Task: predict product `average_rating` (regression) from metadata.

Baseline: Linear Regression → MLP (Keras).

In [ ]:
import os
import sys
from pathlib import Path

for path in [
    Path("/content/smart-product-intelligence"),
    Path("/content/drive/MyDrive/smart-product-intelligence"),
    Path.cwd(),
    Path.cwd().parent,
]:
    if (path / "src" / "data.py").exists():
        os.chdir(path)
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))
        print("Project root:", path)
        break

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import aggregate_reviews_for_products, artifacts_dir, load_splits, prepare_tabular_features
from src.models import (
    build_linear_regression_pipeline,
    build_mlp_regressor,
    regression_metrics,
    set_global_seed,
)
from src.utils import plot_learning_curves, save_json, setup_notebook_path

setup_notebook_path()
set_global_seed(42)

In [ ]:
splits = load_splits()
train_p = splits["train_products"].copy()
val_p = splits["val_products"].copy()
test_p = splits["test_products"].copy()

all_reviews = pd.concat(
    [splits["train_reviews"], splits["val_reviews"], splits["test_reviews"]],
    ignore_index=True,
)
agg = aggregate_reviews_for_products(all_reviews)
train_p = train_p.merge(agg, on="product_id", how="left")
val_p = val_p.merge(agg, on="product_id", how="left")
test_p = test_p.merge(agg, on="product_id", how="left")

train_p, _ = prepare_tabular_features(train_p)
val_p, _ = prepare_tabular_features(val_p)
test_p, _ = prepare_tabular_features(test_p)

numeric = ["price", "log_review_count", "mean_review_rating", "mean_helpful", "mean_review_length"]
categorical = ["main_category", "category_leaf"]
target = "average_rating"

for col in numeric:
    for df in (train_p, val_p, test_p):
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

train_p = train_p.dropna(subset=[target])
val_p = val_p.dropna(subset=[target])
test_p = test_p.dropna(subset=[target])

In [ ]:
X_train, y_train = train_p[numeric + categorical], train_p[target]
X_val, y_val = val_p[numeric + categorical], val_p[target]
X_test, y_test = test_p[numeric + categorical], test_p[target]

baseline = build_linear_regression_pipeline(numeric, categorical)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_metrics = regression_metrics(y_test, baseline_pred)
print("Baseline:", baseline_metrics)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_train = enc.fit_transform(X_train[categorical])
cat_val = enc.transform(X_val[categorical])
cat_test = enc.transform(X_test[categorical])

num_train = X_train[numeric].values.astype(np.float32)
num_val = X_val[numeric].values.astype(np.float32)
num_test = X_test[numeric].values.astype(np.float32)

X_train_mlp = np.hstack([num_train, cat_train]).astype(np.float32)
X_val_mlp = np.hstack([num_val, cat_val]).astype(np.float32)
X_test_mlp = np.hstack([num_test, cat_test]).astype(np.float32)

mean = X_train_mlp.mean(axis=0)
std = X_train_mlp.std(axis=0)
X_train_mlp = (X_train_mlp - mean) / (std + 1e-8)
X_val_mlp = (X_val_mlp - mean) / (std + 1e-8)
X_test_mlp = (X_test_mlp - mean) / (std + 1e-8)

mlp = build_mlp_regressor(X_train_mlp.shape[1])
history = mlp.fit(
    X_train_mlp,
    y_train.values,
    validation_data=(X_val_mlp, y_val.values),
    epochs=30,
    batch_size=64,
    callbacks=[
        __import__("tensorflow").keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ],
    verbose=1,
)
mlp_pred = mlp.predict(X_test_mlp, verbose=0).reshape(-1)
mlp_metrics = regression_metrics(y_test, mlp_pred)
print("MLP:", mlp_metrics)

In [ ]:
plot_learning_curves(history, "Tabular MLP", artifacts_dir() / "m1_learning_curves.png")

ckpt = artifacts_dir().parent / "checkpoints"
ckpt.mkdir(parents=True, exist_ok=True)
mlp.save(ckpt / "tabular_mlp.keras")

import joblib

joblib.dump(
    {"numeric": numeric, "categorical": categorical, "encoder": enc, "mean": mean, "std": std},
    ckpt / "tabular_preprocessor.joblib",
)

save_json({"baseline": baseline_metrics, "mlp": mlp_metrics}, artifacts_dir() / "tabular_metrics.json")